# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!pip install -qU "langchain>=1.0" "langchain-core>=0.3" "langchain-community>=0.4" "langchain-classic>=0.3" langchain-openai langchain-elasticsearch tiktoken jq lark elasticsearch

In [4]:
!pip install -qU "numpy<2" "langchain>=1.0" "langchain-core>=0.3" "langchain-community>=0.4" "langchain-classic>=0.3" langchain-openai langchain-elasticsearch tiktoken jq lark elasticsearch


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.15.0 requires ml-dtypes~=0.2.0, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-intel 2.15.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.6 which is incompatible.


In [1]:
import os
from getpass import getpass

# Imports LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.vectorstores.elasticsearch import ElasticsearchStore

# Configuration de la clé API (retire le '#' au début si tu veux l'utiliser ici)
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [2]:
import os
from dotenv import load_dotenv

# Imports LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.vectorstores.elasticsearch import ElasticsearchStore

# 1. Charger le fichier .env
load_dotenv()

# 2. Les clés sont maintenant dans l'environnement !
# Tu n'as plus besoin de getpass

True

In [3]:
# Assure-toi d'avoir aussi ajouté ton Cloud ID dans ton fichier .env comme ceci :
# ELASTIC_CLOUD_ID=ton_cloud_id_ici

ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

INDEX_NAME = "mon_index_nasa" # N'oublie pas de mettre ton nom d'index !

# Ensuite, le reste du code reste le même pour se connecter :
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings
)

C:\Users\rache\AppData\Local\Temp\ipykernel_22804\2943912541.py:13: LangChainPendingDeprecationWarning: The class `ElasticsearchStore` will be deprecated in a future version. Use `Use class in langchain-elasticsearch package` instead.
  vectorstore = ElasticsearchStore(


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [5]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [6]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json  # Optional: for metadata extraction
from datetime import datetime # Import datetime here


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    None
    return metadata



loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",  # Extracts array of records
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,       # e.g., ~750 words
    chunk_overlap=200,     # Overlap for context preservation
)

docs = loader.load_and_split(text_splitter=text_splitter)


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [8]:
import os
from datetime import datetime
from dotenv import load_dotenv

# Imports LangChain
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# 1. On essaie de charger le fichier .env
load_dotenv(override=True)
ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
INDEX_NAME = "mon_index_nasa" 

# --- VÉRIFICATION BLINDÉE ---
if not OPENAI_API_KEY or not OPENAI_API_KEY.startswith("sk-"):
    print("⚠️ Le fichier .env n'a pas été lu correctement pour OpenAI.")
    OPENAI_API_KEY = input("Colle ta clé OpenAI ici (sk-...) et fais Entrée : ")

if not ELASTIC_API_KEY:
    print("⚠️ Le fichier .env n'a pas été lu correctement pour Elasticsearch.")
    ELASTIC_API_KEY = input("Colle ton API KEY Elastic ici et fais Entrée : ")

if not ELASTIC_CLOUD_ID:
    ELASTIC_CLOUD_ID = input("Colle ton CLOUD ID Elastic ici et fais Entrée : ")
# -----------------------------

# 2. Nettoyage des métadonnées
clean_docs = []
for doc in docs:
    metadata = doc.metadata.copy()
    if metadata.get("updated_at") in ["", None, "null"]:
        metadata["updated_at"] = datetime.now().isoformat()
    clean_docs.append(doc.model_copy(update={"metadata": metadata}))

print(f"✅ {len(clean_docs)} documents nettoyés.")

# 3. Création des Embeddings (C'est ici que ça plantait !)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

# 4. Envoi vers Elasticsearch
print("⏳ Envoi des documents vers Elasticsearch (ça peut prendre un peu de temps)...")
vectorstore = ElasticsearchStore.from_documents(
    documents=clean_docs,
    embedding=embeddings,
    index_name=INDEX_NAME,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)
print("✅ Documents indexés avec succès dans Elasticsearch !")

# 5. Création du LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# 6. Création du MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm
)
print("✅ MultiQueryRetriever configuré et prêt à l'emploi !")

✅ 15 documents nettoyés.
⏳ Envoi des documents vers Elasticsearch (ça peut prendre un peu de temps)...
✅ Documents indexés avec succès dans Elasticsearch !
✅ MultiQueryRetriever configuré et prêt à l'emploi !


# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [9]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain as lc_chain
import logging

# Enable detailed logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Multi-query generator with 3 variants
MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Generate 3 diverse versions of this question for better retrieval. Vary phrasing, keywords, and perspectives:

{question}

Queries (one per line):
""")

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template("""
📄 [{source}]
{page_content}
---
""")

# Define the context prompt for the LLM
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def safe_combine_docs(docs):
    """Production-ready doc formatting with fallbacks"""
    doc_strings = []
    for i, doc in enumerate(docs):
        try:
            doc_dict = doc.model_dump()
            source = doc.metadata.get("name") or doc.metadata.get("source", f"Doc-{i}")
            doc_dict["source"] = source
            formatted = LLM_DOCUMENT_PROMPT.format(**doc_dict)
        except Exception as e:
            logger.warning(f"Doc format error: {e}")
            formatted = f"[Doc-{i}] {doc.page_content[:500]}..."
        doc_strings.append(formatted)
    return "\n\n".join(doc_strings)

# Self-healing chain: retry bad retrievals
def self_healing_retriever(query, max_tries=2):
    """Retry with rewritten query if empty results"""
    for attempt in range(max_tries):
        docs = retriever.invoke(query)
        if docs:
            return docs
        logger.info(f"Empty results (attempt {attempt+1}), rewriting...")
        query = llm.invoke(f"Rewrite for better retrieval: {query}").content
    return retriever.invoke(query)  # Fallback

_context = RunnableParallel(
    context=(RunnablePassthrough() | self_healing_retriever | safe_combine_docs),
    question=RunnablePassthrough(),
)

rag_chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

# Test with auto-multi-query
def multi_query_rag(question):
    """Generate + retrieve + answer"""

    query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()

    generated_queries = query_chain.invoke({"question": question})

    print("\nGenerated Queries:")
    print("------------------")
    print(generated_queries)
    print("------------------\n")

    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]

    all_docs = []

    for q in queries:
        docs = self_healing_retriever(q)
        all_docs.extend(docs[:3])

    return rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs)
    })

print("---- Answer ----")

print(multi_query_rag("what is the nasa sales team?"))

---- Answer ----


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Generated Queries:
------------------
1. Can you provide information on the sales team at NASA?
2. What does the sales team at NASA do?
3. How does NASA's sales team operate and contribute to the organization's goals?
------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What details can you share about the sales department within NASA?', '2. Could you give me insights into the sales team operating within NASA?', "3. I'm interested in learning more about the sales team specifically within NASA. Can you provide relevant information?"]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://ae7bcb62d01c4fc89bd0ea70b46d0d02.es.us-central1.gcp.elastic.cloud:443/mon_index_nasa/_search?_source_includes=metadata,text [status:200 duration:0.400s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://ae7bcb62d01c4fc89bd0ea70b46d0d02.es.us-central1.gcp.elastic.cloud:443/mon_index_nasa/_search?_source_includes=metadata,text [status:200 duration:0.152s]
INF

The NASA sales team consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [13]:
# On importe depuis le nouveau module "core"
from langchain_core.prompts import PromptTemplate

print("--- Itération 1 : Modification du Prompt pour des angles spécifiques ---")

# 1. On crée un prompt personnalisé
custom_prompt_template = """Tu es un expert en recherche de documents. 
Génère 3 versions différentes de la question de l'utilisateur pour interroger une base de données.
Tu DOIS générer ces 3 questions en anglais sous 3 angles différents :
1. Un angle très technique ou opérationnel.
2. Un angle business, financier ou managérial.
3. Un angle simplifié pour un nouvel employé.

Question originale : {question}
"""
prompt_perso = PromptTemplate(
    input_variables=["question"],
    template=custom_prompt_template
)

# 2. On crée un nouveau retriever avec ce prompt
retriever_perspectives = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm,
    prompt=prompt_perso
)

# 3. Testons juste la génération de questions pour voir la magie opérer
question_test = "How does the company handle remote work?"
print(f"Question initiale : {question_test}\n")

# L'appel interne modernisé (LangChain Expression Language - LCEL)
chain = prompt_perso | llm
questions_generees = chain.invoke({"question": question_test})

print("Questions générées par le LLM (Les 3 angles) :")
# On affiche le contenu de la réponse du LLM
print(questions_generees.content)

--- Itération 1 : Modification du Prompt pour des angles spécifiques ---
Question initiale : How does the company handle remote work?



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Questions générées par le LLM (Les 3 angles) :
1. Technical/Operational angle: 
- What specific tools and technologies does the company utilize to facilitate remote work for its employees?

2. Business/Financial/Managerial angle:
- How has the implementation of remote work impacted the company's productivity, cost savings, and overall business operations?

3. Simplified angle for a new employee:
- Can you explain how employees at the company are able to work from home and stay connected with their colleagues and managers?


In [11]:
print("--- Itération 2 : LLM Créatif (Temp: 0.8) et Recherche Large (k: 10) ---")

# 1. On crée un nouveau "cerveau" plus créatif (température à 0.8 au lieu de 0)
llm_creatif = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.8, 
    api_key=OPENAI_API_KEY
)

# 2. On crée un retriever qui va chercher beaucoup plus de documents (k=10)
retriever_large = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 10}),
    llm=llm_creatif
)

# 3. On teste avec une question un peu vague pour voir si le retriever arrive à trouver du contenu
question_vague = "Tell me about any workplace conflicts or employee feedback."

# On lance la recherche
documents_trouves = retriever_large.invoke(question_vague)

print(f"Question posée : '{question_vague}'")
print(f"Nombre de documents uniques pertinents récupérés : {len(documents_trouves)}")

# On affiche un extrait du meilleur document trouvé
if documents_trouves:
    print("\nExtrait du document le plus pertinent trouvé :")
    print(f"'{documents_trouves[0].page_content[:200]}...'")
else:
    print("Aucun document trouvé.")

--- Itération 2 : LLM Créatif (Temp: 0.8) et Recherche Large (k: 10) ---


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What are some examples of workplace conflicts that have arisen in a professional setting?', 'Can you provide insights into employee feedback within a work environment?', "I'm interested in learning more about conflicts and feedback among employees in a workplace."]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://ae7bcb62d01c4fc89bd0ea70b46d0d02.es.us-central1.gcp.elastic.cloud:443/mon_index_nasa/_search?_source_includes=metadata,text [status:200 duration:0.925s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://ae7bcb62d01c4fc89bd0ea70b46d0d02.es.us-central1.gcp.elastic.cloud:443/mon_index_nasa/_search?_source_includes=metadata,text [status:200 duration:0.132s]
INFO:htt

Question posée : 'Tell me about any workplace conflicts or employee feedback.'
Nombre de documents uniques pertinents récupérés : 11

Extrait du document le plus pertinent trouvé :
'Starting May 2022, the company will be implementing a two-day in-office work requirement per week for all eligible employees. Please coordinate with your supervisor and HR department to schedule your ...'
